# IPA via three elbow methods — Kneedle, max 2nd derivative, min distance from origin

All methods operate on the raw averaged CE-vs-BN data (step-artifact truncated,
`BN_STEP_MIN=100`, `STEP_THRESH=0.01`); no curve fitting anywhere.
`IPA = |CE_o − CE_learned| / BN_learned`, `CE_o = ln(10)`.

- **Cell 1 — Kneedle** (also defines the shared loader/sweep helpers): `kneed.KneeLocator`,
  optional tail-average anchor via `KNEEDLE_TAIL_N`.
- **Cell 2 — max second derivative**: raw central differences (`SMOOTH_W=1`; odd >1 = moving avg).
- **Cell 3 — min distance from origin**: closest point to the normalized ideal corner.
- **Cell 4 — comparison**: all three methods per batch size (log y), plus numeric tables.

Each method writes `ipa_summary_method_*.csv` and `ipa_plot_method_*.png`.
Requires `pip install kneed`. Run cells top to bottom (cells 2-4 use Cell 1 helpers).

In [5]:
# ============================================================================
# Cell 1 — KNEEDLE METHOD  (also defines shared helpers used by cells 2-4)
#
# Elbow via kneed.KneeLocator (curve='convex', direction='decreasing').
# KNEEDLE_TAIL_N: None = standard min(y) anchor; int = mean of last N points.
# IPA = |CE_o - CE_learned| / BN_learned
# ============================================================================
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BN_STEP_MIN = 100     # step-artifact detection (shared convention)
STEP_THRESH = 0.01
CE_o = np.log(10)     # ~2.302585
BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
OUT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step"
BATCH_SIZES = [64, 1024, 60000]
BS_COLOR = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

PRUNING_LEVELS = sorted(float(re.search(r"p-percentage_([\d.]+)", d).group(1))
                        for d in glob.glob(os.path.join(BASE_DIR, "p-percentage_*")))


def load_curve(p, bs):
    """Averaged CE curve for (p, bs), truncated at the step artifact.
    Returns (BN, CE) arrays or None."""
    f = os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                     f"averaged_runs_p_{p}_bs_{bs}.csv")
    if not os.path.exists(f):
        return None
    df = pd.read_csv(f)
    df.columns = df.columns.str.strip()
    ce_col = next((c for c in df.columns if c in ("Avg_CE_Test", "Avg_CE_test")), None)
    bn_col = next((c for c in df.columns if "Batch" in c), None)
    if ce_col is None or bn_col is None:
        return None
    df = df.dropna(subset=[ce_col, bn_col])
    bns = df[bn_col].values.astype(float)
    ces = df[ce_col].values.astype(float)
    cutoff_BN = float(bns[-1])
    for i in range(1, len(bns)):
        if bns[i] >= BN_STEP_MIN and abs(ces[i] - ces[i - 1]) > STEP_THRESH:
            cutoff_BN = float(bns[i])
            break
    m = bns < cutoff_BN
    return bns[m], ces[m]


def sweep_and_plot(elbow_fn, method_name, file_tag, marker="o-"):
    """Run elbow_fn(BN, CE) -> (BN_learned, CE_learned, IPA) over all (P%, BS);
    save summary CSV + IPA-vs-P% plot; return the summary DataFrame."""
    rows = []
    for p in PRUNING_LEVELS:
        row = {"P%": p * 100}
        for bs in BATCH_SIZES:
            curve = load_curve(p, bs)
            bn_l, ce_l, ipa = elbow_fn(*curve) if curve is not None else (np.nan,)*3
            row[f"BN_learned_{bs}"] = bn_l
            row[f"IPA_Avg_{bs}"]    = ipa
        rows.append(row)
    summary = pd.DataFrame(rows)

    csv_path = os.path.join(OUT_DIR, f"ipa_summary_{file_tag}.csv")
    summary.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")
    print(summary[["P%"] + [f"IPA_Avg_{b}" for b in BATCH_SIZES]].to_string(index=False))

    plt.rcParams.update({"font.size": 13})
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for bs in BATCH_SIZES:
        sub = summary.dropna(subset=[f"IPA_Avg_{bs}"])
        ax.plot(sub["P%"], sub[f"IPA_Avg_{bs}"], marker, color=BS_COLOR[bs],
                ms=5, lw=2, label=f"BS={bs}")
    ax.set_xlabel("Pruning Percentage (%)")
    ax.set_ylabel("IPA")
    ax.set_title(f"IPA vs Pruning — {method_name}")
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False)
    png_path = os.path.join(OUT_DIR, f"ipa_plot_{file_tag}.png")
    plt.tight_layout(); plt.savefig(png_path, dpi=150, bbox_inches="tight"); plt.close(fig)
    print(f"Saved: {png_path}")
    return summary

print(f"Common helpers ready — {len(PRUNING_LEVELS)} pruning levels")

# ── Kneedle method ────────────────────────────────────────────────────────────
from kneed import KneeLocator

KNEEDLE_S      = 1.0
KNEEDLE_TAIL_N = None    # None = standard min(y) anchor; int = mean of last N

def kneedle_elbow(BN, CE, S=KNEEDLE_S, tail_n=KNEEDLE_TAIL_N):
    BN = np.asarray(BN, float); CE = np.asarray(CE, float)
    if len(BN) < 3 or np.ptp(CE) <= 1e-10:
        return np.nan, np.nan, np.nan
    if tail_n is not None:
        k = int(min(tail_n, len(CE)))
        floor = float(np.mean(CE[-k:]))
        if CE[0] - floor <= 1e-10:
            return np.nan, np.nan, np.nan
        y = np.clip(CE, floor, None)
    else:
        y = CE
    kl = KneeLocator(BN, y, curve="convex", direction="decreasing", S=S)
    if kl.knee is None:
        return np.nan, np.nan, np.nan
    i = int(np.argmin(np.abs(BN - kl.knee)))
    if BN[i] <= 0:
        return np.nan, np.nan, np.nan
    return float(BN[i]), float(CE[i]), abs(CE_o - CE[i]) / BN[i]

kneedle_summary_df = sweep_and_plot(kneedle_elbow, "Kneedle method", "method_kneedle", "s--")


Common helpers ready — 19 pruning levels
Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_summary_method_kneedle.csv
   P%  IPA_Avg_64  IPA_Avg_1024  IPA_Avg_60000
  0.0    0.054735      0.067119       0.076981
 10.0    0.047737      0.062432       0.073748
 20.0    0.048365      0.058271       0.072946
 30.0    0.048767      0.055950       0.069466
 40.0    0.040060      0.056493       0.061973
 50.0    0.035555      0.047364       0.057097
 60.0    0.032051      0.041137       0.047341
 70.0    0.025533      0.034859       0.038163
 80.0    0.020862      0.026574       0.030385
 82.0    0.018689      0.025826       0.027507
 84.0    0.017850      0.024001       0.025712
 86.0    0.016741      0.021854       0.023611
 88.0    0.014170      0.019878       0.022110
 90.0    0.013475      0.018431       0.019321
 92.0    0.011734      0.015982       0.016624
 94.0    0.010137      0.013527       0.014321
 96.0

In [6]:
# ============================================================================
# Cell 2 — MAX SECOND DERIVATIVE METHOD  (raw data, no fitting)
#
# Elbow = data point with maximum discrete second derivative
#     d2_i ~ CE_{i+1} - 2*CE_i + CE_{i-1}   (central differences, np.gradient)
# SMOOTH_W = 1 uses the raw data; odd int > 1 applies a moving average first.
# ============================================================================
SMOOTH_W = 30

def second_derivative_elbow(BN, CE, smooth_w=SMOOTH_W):
    BN = np.asarray(BN, float); CE = np.asarray(CE, float)
    if len(BN) < max(5, smooth_w + 2) or np.ptp(CE) <= 1e-10:
        return np.nan, np.nan, np.nan
    if smooth_w > 1:
        y = np.convolve(CE, np.ones(int(smooth_w)) / smooth_w, mode="same")
        h = int(smooth_w) // 2
    else:
        y, h = CE, 0
    d2 = np.gradient(np.gradient(y, BN), BN)
    lo, hi = 1 + h, len(BN) - 1 - h
    if hi <= lo:
        return np.nan, np.nan, np.nan
    i = lo + int(np.argmax(d2[lo:hi]))
    if BN[i] <= 0:
        return np.nan, np.nan, np.nan
    return float(BN[i]), float(CE[i]), abs(CE_o - CE[i]) / BN[i]

deriv_summary_df = sweep_and_plot(second_derivative_elbow,
                                  "max 2nd derivative (raw data)",
                                  "method_2nd_derivative", "^-.")


Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_summary_method_2nd_derivative.csv
   P%  IPA_Avg_64  IPA_Avg_1024  IPA_Avg_60000
  0.0    0.092279      0.099318       0.098507
 10.0    0.090730      0.098041       0.097699
 20.0    0.088628      0.096691       0.096563
 30.0    0.086303      0.094777       0.094810
 40.0    0.083339      0.092203       0.092512
 50.0    0.078459      0.088630       0.089543
 60.0    0.072497      0.083310       0.084644
 70.0    0.063693      0.074687       0.077215
 80.0    0.050539      0.061598       0.064063
 82.0    0.047569      0.058118       0.060640
 84.0    0.043501      0.053736       0.055829
 86.0    0.039032      0.049073       0.051762
 88.0    0.034991      0.044222       0.046660
 90.0    0.030328      0.038866       0.040867
 92.0    0.025286      0.032621       0.033524
 94.0    0.019378      0.025649       0.027048
 96.0    0.013664      0.017761       0

In [7]:
# ============================================================================
# Cell 3 — MIN DISTANCE FROM ORIGIN METHOD
#
# Normalize the window, elbow = data point closest to the origin (0,0)
# (= earliest BN, fully converged CE):  argmin( x_hat^2 + y_hat^2 )
#   x_hat = (BN - BN[0]) / (BN[-1] - BN[0])
#   y_hat = (CE - A) / (CE[0] - A),  A = min(CE) or mean of last ORIGIN_TAIL_N
# ============================================================================
ORIGIN_TAIL_N = None    # None = A = min(CE); int = A = mean of last N points

def origin_elbow(BN, CE, tail_n=ORIGIN_TAIL_N):
    BN = np.asarray(BN, float); CE = np.asarray(CE, float)
    if len(BN) < 3 or np.ptp(CE) <= 1e-10:
        return np.nan, np.nan, np.nan
    A = float(np.mean(CE[-int(min(tail_n, len(CE))):])) if tail_n is not None else float(np.min(CE))
    BN_range = BN[-1] - BN[0]
    CE_range = CE[0] - A
    if BN_range <= 0 or CE_range <= 1e-10:
        return np.nan, np.nan, np.nan
    x_hat = (BN - BN[0]) / BN_range
    y_hat = (CE - A) / CE_range
    i = int(np.argmin(x_hat**2 + y_hat**2))
    if BN[i] <= 0:
        return np.nan, np.nan, np.nan
    return float(BN[i]), float(CE[i]), abs(CE_o - CE[i]) / BN[i]

origin_summary_df = sweep_and_plot(origin_elbow, "min distance from origin",
                                   "method_origin", "o-")


Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_summary_method_origin.csv
   P%  IPA_Avg_64  IPA_Avg_1024  IPA_Avg_60000
  0.0    0.057926      0.067119       0.082997
 10.0    0.054202      0.062432       0.079298
 20.0    0.052078      0.063713       0.075583
 30.0    0.049966      0.055950       0.071863
 40.0    0.044507      0.054963       0.061973
 50.0    0.039022      0.047364       0.057097
 60.0    0.032051      0.041137       0.047341
 70.0    0.026917      0.036108       0.038163
 80.0    0.021318      0.027689       0.030852
 82.0    0.019234      0.026521       0.028287
 84.0    0.018892      0.024296       0.026051
 86.0    0.017505      0.022358       0.024193
 88.0    0.015459      0.020524       0.022629
 90.0    0.013886      0.018801       0.019726
 92.0    0.012225      0.016273       0.017098
 94.0    0.010462      0.013980       0.014693
 96.0    0.008081      0.010567       0.011183


In [8]:
# ============================================================================
# Cell 4 — COMPARISON: all three methods, one panel per batch size
#
# Uses the in-memory summaries from cells 1-3 (falls back to their CSVs).
# y-axis is LOG scale: the 2nd-derivative method picks very early elbows,
# so its IPA is an order of magnitude above the other two.
# ============================================================================
if "kneedle_summary_df" not in dir():
    kneedle_summary_df = pd.read_csv(os.path.join(OUT_DIR, "ipa_summary_method_kneedle.csv"))
if "deriv_summary_df" not in dir():
    deriv_summary_df = pd.read_csv(os.path.join(OUT_DIR, "ipa_summary_method_2nd_derivative.csv"))
if "origin_summary_df" not in dir():
    origin_summary_df = pd.read_csv(os.path.join(OUT_DIR, "ipa_summary_method_origin.csv"))

METHODS = [("Kneedle",               kneedle_summary_df, "#ff7f0e", "s--"),
           ("max 2nd derivative",    deriv_summary_df,   "#9467bd", "^-."),
           ("min dist. from origin", origin_summary_df,  "#2ca02c", "o-")]

plt.rcParams.update({"font.size": 13})
fig, axes = plt.subplots(1, len(BATCH_SIZES), figsize=(6 * len(BATCH_SIZES), 5),
                         sharex=True, sharey=True)
if len(BATCH_SIZES) == 1:
    axes = [axes]
for ax, bs in zip(axes, BATCH_SIZES):
    col = f"IPA_Avg_{bs}"
    for name, df_m, color, style in METHODS:
        sub = df_m.dropna(subset=[col])
        ax.plot(sub["P%"], sub[col], style, color=color, ms=5, lw=1.8, label=name)
    ax.set_yscale("log")
    ax.set_title(f"BS={bs}")
    ax.set_xlabel("Pruning Percentage (%)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(frameon=False, fontsize=9)
axes[0].set_ylabel("IPA (log scale)")
fig.suptitle("IPA vs Pruning — Kneedle vs max 2nd derivative vs min distance from origin",
             fontsize=13)
out_png = os.path.join(OUT_DIR, "ipa_plot_compare_three_methods.png")
plt.tight_layout()
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_png}")

# Numeric side-by-side (per BS)
for bs in BATCH_SIZES:
    col = f"IPA_Avg_{bs}"
    cmp_df = pd.DataFrame({"P%": kneedle_summary_df["P%"],
                           "kneedle": kneedle_summary_df[col],
                           "2nd_deriv": deriv_summary_df[col],
                           "origin": origin_summary_df[col]})
    print(f"\n=== BS={bs} ===")
    print(cmp_df.to_string(index=False))


Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_plot_compare_three_methods.png

=== BS=64 ===
   P%  kneedle  2nd_deriv   origin
  0.0 0.054735   0.092279 0.057926
 10.0 0.047737   0.090730 0.054202
 20.0 0.048365   0.088628 0.052078
 30.0 0.048767   0.086303 0.049966
 40.0 0.040060   0.083339 0.044507
 50.0 0.035555   0.078459 0.039022
 60.0 0.032051   0.072497 0.032051
 70.0 0.025533   0.063693 0.026917
 80.0 0.020862   0.050539 0.021318
 82.0 0.018689   0.047569 0.019234
 84.0 0.017850   0.043501 0.018892
 86.0 0.016741   0.039032 0.017505
 88.0 0.014170   0.034991 0.015459
 90.0 0.013475   0.030328 0.013886
 92.0 0.011734   0.025286 0.012225
 94.0 0.010137   0.019378 0.010462
 96.0 0.007944   0.013664 0.008081
 98.0 0.004837   0.006863 0.004882
100.0      NaN        NaN      NaN

=== BS=1024 ===
   P%  kneedle  2nd_deriv   origin
  0.0 0.067119   0.099318 0.067119
 10.0 0.062432   0.098041 0.062432
 20.